# RAG Query Pipeline

Load the vector store built in `01_ingest.ipynb`, retrieve relevant chunks, generate answer (text + tables) with Groq.

In [ ]:
# Imports + load API keys from .env - shared pipeline logic lives in rag_pipeline.py
import json

from dotenv import load_dotenv

import rag_pipeline

load_dotenv()

llm = rag_pipeline.get_llm()
embedding_model = rag_pipeline.get_embedding_model()

In [ ]:
# Load the persisted vector store created in 01_ingest.ipynb
# Must use the SAME embedding model as ingest, or similarity search breaks
persist_directory = "dbv2/chroma_db"  # match whichever dir you used at ingest time

db = rag_pipeline.load_vector_store(persist_directory, embedding_model)

In [ ]:
# Helper: dump retrieved chunks to a JSON file for inspection
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []

    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

In [ ]:
# Retrieve - uses rag_pipeline.retrieve_chunks
query = "What are the two main components of the Transformer architecture? "
chunks = rag_pipeline.retrieve_chunks(db, query, k=3)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

In [ ]:
# Step: generate final answer from retrieved chunks using Groq - uses rag_pipeline.generate_final_answer, then test it end-to-end
query = "How many attention heads does the Transformer use, and what is the dimension of each head? "
chunks = rag_pipeline.retrieve_chunks(db, query, k=3)

final_answer = rag_pipeline.generate_final_answer(llm, chunks, query)
print(final_answer)